In [1]:
import pandas as pd
import json

from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR,NuSVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd
from helpers.modeling import (
    identify_column_types,
    create_preprocessor,
    evaluate_model,
)


with open('results.json', 'r') as f:
    results = json.load(f)

In [2]:
df = pd.read_csv("../Datasets/processed/UHPC_dataset/semantic_recoding_features_50_with_publications.csv")
df.head()

,cement,cement_type,silica_fume,fly_ash,fly_ash_type,limestone_powder,quartz_powder,glass_powder,rice_husk_ash,metakaolin,...,fiber1_diameter,fiber2_type,fiber2_amount,water,sp_type,sp_amount,curing_method,curing_temp,cs_28d,paper_reference
0,839.0,OPC_I,104.0,104.0,class F,0.0,0.0,0.0,0.0,0.0,...,0.2,not_applicable,0.0,147.0,PCE_SP,56.50,Standard Curing,NaN,135.0,Ref-1-data
1,839.0,OPC_I,104.0,52.0,class F,0.0,0.0,0.0,0.0,0.0,...,0.2,not_applicable,0.0,147.0,PCE_SP,59.33,Standard Curing,NaN,132.0,Ref-1-data
2,839.0,OPC_I,104.0,26.0,class F,0.0,0.0,0.0,0.0,0.0,...,0.2,not_applicable,0.0,147.0,PCE_SP,59.33,Standard Curing,NaN,122.5,Ref-1-data
3,839.0,OPC_I,104.0,0.0,not_applicable,0.0,0.0,0.0,0.0,0.0,...,0.2,not_applicable,0.0,147.0,PCE_SP,62.15,Standard Curing,NaN,116.0,Ref-1-data
4,839.0,OPC_I,104.0,52.0,class F,0.0,0.0,0.0,0.0,52.0,...,0.2,not_applicable,0.0,147.0,PCE_SP,64.98,Standard Curing,NaN,134.0,Ref-1-data


In [3]:
X = df.drop(columns="cs_28d")
y = df["cs_28d"]

pub_col = "paper_reference"  
held_out_pub = "Ref-1-data"

train_mask = df[pub_col] != held_out_pub
test_mask  = df[pub_col] == held_out_pub

X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

In [4]:
numerical_cols, one_hot_columns, k_fold_columns = identify_column_types(X)

preprocessor = create_preprocessor(numerical_cols, one_hot_columns, k_fold_columns, 
                                   handle_unknown='ignore')

In [5]:
def run_pipeline(model_cls, model_key, kernel_kwargs=None):
    params = results["best_params"]["best_params_publications_included"][model_key]
    pipeline = Pipeline([('preprocessor', preprocessor),
                          ('model', model_cls(**(kernel_kwargs or {})))])
    pipeline.set_params(**params)
    pipeline.fit(X_train, y_train)

    train_metrics = evaluate_model(y_train, pipeline.predict(X_train), set_name='train')
    test_metrics = evaluate_model(y_test, pipeline.predict(X_test), set_name='test')
    return pipeline, train_metrics, test_metrics

In [6]:
knn_pipeline, knn_train, knn_test = run_pipeline(KNeighborsRegressor, "knn")


c:\Users\shoai\anaconda3\envs\aicome\Lib\site-packages\threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)



TRAIN SET PERFORMANCE
RMSE: 6.3051
MAE: 3.9230
R2: 0.9700
Correlation: 0.9851
Mean_Residual: -0.0730
N: 2057

TEST SET PERFORMANCE
RMSE: 10.6068
MAE: 8.1000
R2: -1.0191
Correlation: 0.2068
Mean_Residual: -5.8484
N: 16


In [7]:
svr_pipeline, svr_train, svr_test = run_pipeline(SVR, "svr", {"kernel": "rbf"})



TRAIN SET PERFORMANCE
RMSE: 7.6127
MAE: 4.9531
R2: 0.9563
Correlation: 0.9780
Mean_Residual: -0.0727
N: 2057

TEST SET PERFORMANCE
RMSE: 26.7773
MAE: 24.1042
R2: -11.8687
Correlation: -0.6036
Mean_Residual: -24.1042
N: 16


In [8]:
nusvr_pipeline, nusvr_train, nusvr_test = run_pipeline(NuSVR, "nusvr", {"kernel": "rbf"})


TRAIN SET PERFORMANCE
RMSE: 8.2474
MAE: 5.6453
R2: 0.9487
Correlation: 0.9742
Mean_Residual: -0.1728
N: 2057

TEST SET PERFORMANCE
RMSE: 27.1407
MAE: 25.8961
R2: -12.2203
Correlation: -0.3390
Mean_Residual: -25.8961
N: 16
